# 09c — US regime-intercept correction

This notebook corrects the US regime diagnostic in Table 8 by allowing a separate intercept in each regime. The panel and Fama--MacBeth rows are unchanged. The input is the frozen model-ready G10 panel from notebook 09 v7.

In [1]:

import hashlib, pandas as pd, numpy as np
from pathlib import Path
from statistics import NormalDist
INPUT = Path("g10_panel_model_objects.csv")
if not INPUT.exists():
    hits = list(Path('.').rglob(INPUT.name))
    if not hits:
        raise FileNotFoundError(INPUT.name)
    INPUT = hits[0]
EXPECTED = "1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6"
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20), b''): h.update(block)
    return h.hexdigest()
got=sha256(INPUT)
assert got==EXPECTED, (got, EXPECTED)
print('input SHA-256 verified:',got)
p=pd.read_csv(INPUT,parse_dates=['date'])
usa=p.loc[p['country'].eq('USA')].dropna(subset=['rx','tp_z_past']).copy()
print('USA rows:',len(usa),'|',usa.date.min().date(),'->',usa.date.max().date())


input SHA-256 verified: 1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6
USA rows: 478 | 1984-06-30 -> 2024-04-30


In [2]:

REGIME_ORDER=['pre_2008','2008_2015','2016_plus']
def regime_name(date):
    date=pd.Timestamp(date)
    if date<=pd.Timestamp('2007-12-31'): return 'pre_2008'
    if date<=pd.Timestamp('2015-12-31'): return '2008_2015'
    return '2016_plus'
usa['regime']=usa['date'].map(regime_name)
def nw_fit(y,X,lag=12):
    y=np.asarray(y,float); X=np.asarray(X,float)
    beta=np.linalg.lstsq(X,y,rcond=None)[0]
    e=y-X@beta
    bread=np.linalg.pinv(X.T@X)
    score=X*e[:,None]
    S=score.T@score
    for ell in range(1,min(lag,len(y)-1)+1):
        w=1.0-ell/(lag+1.0)
        G=score[ell:].T@score[:-ell]
        S += w*(G+G.T)
    V=bread@S@bread
    se=np.sqrt(np.maximum(np.diag(V),0.0))
    return beta,se,V
I=np.column_stack([(usa['regime']==r).astype(float).to_numpy() for r in REGIME_ORDER])
S=np.column_stack([usa['tp_z_past'].to_numpy()*(usa['regime']==r).astype(float).to_numpy() for r in REGIME_ORDER])
X=np.column_stack([I,S])
b,se,V=nw_fit(usa['rx'].to_numpy(),X,12)
rows=[]
for j,r in enumerate(REGIME_ORDER):
    k=3+j
    rows.append({'regime':r,'intercept':b[j],'intercept_se':se[j],
                 'beta':b[k],'se':se[k],'t':b[k]/se[k]})
slopes=pd.DataFrame(rows)
display(slopes)
nd=NormalDist(); tests=[]
for a,bix in [(0,1),(0,2),(1,2)]:
    ia=3+a; ib=3+bix
    diff=b[ia]-b[ib]
    var=V[ia,ia]+V[ib,ib]-2*V[ia,ib]
    z=diff/np.sqrt(var)
    pv=2*(1-nd.cdf(abs(z)))
    tests.append({'contrast':f'{REGIME_ORDER[a]} - {REGIME_ORDER[bix]}','diff':diff,'z':z,'p_2s':pv})
tests=pd.DataFrame(tests)
display(tests)


,regime,intercept,intercept_se,beta,se,t
0,pre_2008,0.037431,0.010742,0.104590,0.020727,5.046114
1,2008_2015,0.052731,0.011238,0.070452,0.017892,3.937712
2,2016_plus,-0.004529,0.026369,0.047957,0.025053,1.914230


,contrast,diff,z,p_2s
0,pre_2008 - 2008_2015,0.034138,1.412569,0.157783
1,pre_2008 - 2016_plus,0.056634,1.741748,0.081553
2,2008_2015 - 2016_plus,0.022495,0.732201,0.464046


In [3]:

# Hard checks for the manuscript values.
target=np.array([0.10459035244526474,0.07045205984388814,0.047956821060189087])
assert np.max(np.abs(slopes['beta'].to_numpy()-target)) < 1e-12
ptarget=np.array([0.15778255010848818,0.0815526180722006,0.4640456301999847])
assert np.max(np.abs(tests['p_2s'].to_numpy()-ptarget)) < 1e-12
print('SELF-TEST OK — regime-specific-intercept results reproduce manuscript values.')


SELF-TEST OK — regime-specific-intercept results reproduce manuscript values.
